In [0]:
from pyspark.sql import functions as F

# Πάρε το ήδη-φορτωμένο bronze table
df = spark.table("workspace.aml_bronze.raw_transactions")

# Σπάσε το σε 5 κομμάτια, σαν να έφταναν σε 5 διαφορετικά "batches"
chunks = df.randomSplit([1.0]*5, seed=42)

landing_path = "/Volumes/workspace/aml_raw/streaming_landing"

for i, chunk in enumerate(chunks):
    (
        chunk
        .drop("_ingested_at", "_source_file")  # καθάρισε metadata columns πριν ξαναγράψεις
        .write
        .format("json")
        .mode("overwrite")
        .save(f"{landing_path}/chunk_{i}")
    )
    print(f"Wrote chunk_{i}: {chunk.count():,} rows")

In [0]:
streaming_schema = df.drop("_ingested_at", "_source_file").schema

streaming_df = (
    spark.readStream
    .format("json")
    .schema(streaming_schema)
    .load(landing_path + "/*")
)

In [0]:
checkpoint_path = "/Volumes/workspace/aml_raw/streaming_checkpoint"

query = (
    streaming_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("workspace.aml_bronze.streaming_transactions")
)

query.awaitTermination()
print("Streaming query finished processing available data.")

In [0]:
result = spark.table("workspace.aml_bronze.streaming_transactions")
print(f"Total rows in streaming table: {result.count():,}")

In [0]:
query2 = (
    streaming_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("workspace.aml_bronze.streaming_transactions")
)

query2.awaitTermination()

result2 = spark.table("workspace.aml_bronze.streaming_transactions")
print(f"Total rows after re-running: {result2.count():,}")